In [0]:
# Create a text widgets
dbutils.widgets.text("landing_timestamp", "", "2.Landing Timestamp")
dbutils.widgets.text("table_name", "", "1.Table Name")
dbutils.widgets.text("merge_keys", "", "3.Merge Keys")

# Create a text widget for the load type: FULL, APPEND or MERGE
dbutils.widgets.text("load_type", "", "4.Load Type")

# Read the values from its widgets
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")

# Read the load type value from its widget
load_type = dbutils.widgets.get("load_type")


In [0]:
# Read the bronze table and keep only the rows from the selected landing batch
bronze_df = spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

# Print the equivalent SQL of the bronze read so it is clear which batch is loaded
print("=" * 80)
print("STEP 2: Read bronze batch")
print(f"Query  : SELECT * FROM adb_caresync.bronze.{table_name} WHERE landing_timestamp = '{landing_timestamp}'")
# Print the source columns that were read
print(f"Columns: {bronze_df.columns}")

In [0]:
# Import the function that returns the current date and time
from pyspark.sql.functions import current_timestamp

# Add insert_timestamp and update_timestamp audit columns, both set to the current time
updated_df = bronze_df.withColumn("insert_timestamp", current_timestamp()).withColumn("update_timestamp", current_timestamp())

# Print what this step added
print("=" * 80)
print("STEP 3: Add audit columns")
print("Added  : insert_timestamp = current_timestamp(), update_timestamp = current_timestamp()")
print(f"Columns: {updated_df.columns}")

In [0]:
# Build the full name of the silver target table (catalog.schema.table)
target_table = f"adb_caresync.silver.{table_name}"

# Use FULL when load_type is empty, remove spaces and convert to upper case
load_type_normalized = (load_type or "FULL").strip().upper()

# List of load types this notebook supports
supported_load_types = ["FULL", "APPEND", "MERGE"]

# Check that the given load type is one of the supported ones
if load_type_normalized not in supported_load_types:
    # Stop the notebook with a clear error if the load type is not supported
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: {supported_load_types}")

# Check whether the silver target table already exists (True/False)
target_exists = spark.catalog.tableExists(target_table)

# Start with an empty list of merge keys (only filled for MERGE)
keys = []

# Merge-specific checks run only when the load type is MERGE
if load_type_normalized == "MERGE":
    # Split the comma-separated merge keys into a list and remove blanks/spaces
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    # Check that at least one merge key was provided
    if not keys:
        # Stop the notebook because MERGE cannot run without keys
        raise ValueError("merge_keys is required (comma-separated) when load_type = 'MERGE'")
    # Find any merge keys that are not columns in the source data
    missing_keys = [k for k in keys if k not in updated_df.columns]
    # Check whether any merge key is missing from the source data
    if missing_keys:
        # Stop the notebook and list the missing key columns
        raise ValueError(f"Merge keys not found in source data: {missing_keys}")

# Print a header for this step
print("=" * 80)
print("STEP 4: Validate load parameters - all checks passed")
# Print the target table name and whether it already exists
print(f"Target table : {target_table} (exists: {target_exists})")
# Print the load type that will be used
print(f"Load type    : {load_type_normalized}")
# Print the merge keys, or n/a when the load type is not MERGE
print(f"Merge keys   : {keys if keys else 'n/a'}")

In [0]:
# Import the DeltaTable class, needed to run a MERGE on a Delta table
from delta.tables import DeltaTable
# Import datetime to print the start and end time of the write
from datetime import datetime

# Source query, used in the printed SQL statements below
source_query = f"SELECT * FROM adb_caresync.bronze.{table_name} WHERE landing_timestamp = '{landing_timestamp}'"
# Build the exact column list being written: source columns plus the two audit columns set to current_timestamp()
audit_cols = ["insert_timestamp", "update_timestamp"]
select_list = ", ".join([f"`{c}`" for c in updated_df.columns if c not in audit_cols]
                        + ["current_timestamp() AS insert_timestamp", "current_timestamp() AS update_timestamp"])
# SELECT part shared by the printed FULL / APPEND / first-MERGE statements
select_sql = f"SELECT {select_list}\n  FROM ({source_query})"

# Print a header with the run details
print("=" * 80)
print("STEP 5: Write to silver table")
print(f"Started at   : {datetime.now()}")
print(f"Target table : {target_table}")
print(f"Table exists : {target_exists}")
print(f"Load type    : {load_type_normalized}")

# Count the source rows so the output shows how much data is being written
source_count = updated_df.count()
print(f"Source rows  : {source_count}")
print("-" * 80)

# All writes to the silver table happen in this cell
# FULL load: replace all data in the target table
if load_type_normalized == "FULL":
    # Explain the action and print the equivalent SQL
    print("Action : FULL load - the target table will be overwritten with the new batch")
    print("Equivalent SQL:")
    print(f"  CREATE OR REPLACE TABLE {target_table} AS\n  {select_sql}")
    # Overwrite the target table with the new batch (creates it if missing)
    updated_df.write.mode("overwrite").saveAsTable(target_table)

# APPEND load: add new rows without touching existing ones
elif load_type_normalized == "APPEND":
    # Explain the action and print the equivalent SQL
    print("Action : APPEND load - new rows will be added, existing rows stay unchanged")
    if not target_exists:
        print("Note   : target table does not exist yet, it will be created")
    print("Equivalent SQL:")
    print(f"  INSERT INTO {target_table}\n  {select_sql}")
    # Append the new batch to the target table (creates it if missing)
    updated_df.write.mode("append").saveAsTable(target_table)

# MERGE load: update matching rows and insert new rows
elif load_type_normalized == "MERGE":
    print(f"Action : MERGE load on keys {keys} - matching rows are updated, new rows are inserted")
    # Keep one row per merge key, because MERGE fails if several source rows match one target row
    source_df = updated_df.dropDuplicates(keys)
    # Count rows after removing duplicates and print how many were dropped
    dedup_count = source_df.count()
    print(f"Rows after removing duplicate keys : {dedup_count} (dropped {source_count - dedup_count})")

    # If the target table does not exist yet, there is nothing to merge into
    if not target_exists:
        print("Note   : target table does not exist yet, it will be created from the source data (no merge needed)")
        print("Equivalent SQL:")
        print(f"  CREATE TABLE {target_table} AS\n  {select_sql}")
        # Create the target table from the source data on the first run
        source_df.write.mode("overwrite").saveAsTable(target_table)
    # The target table exists, so run the MERGE
    else:
        # Build the join condition on all merge keys, e.g. t.`id` = s.`id` AND t.`region` = s.`region`
        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])
        # Map each target column to the source value, skipping insert_timestamp so it keeps its original value
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}

        # Print the pieces of the MERGE that were just prepared
        print(f"Merge condition : {merge_condition}")
        print(f"Columns updated : {list(update_set.keys())}")
        print("Column kept     : `insert_timestamp` (not updated on matched rows)")
        # Print the full equivalent MERGE statement
        set_clause = ",\n    ".join([f"t.{k} = {v}" for k, v in update_set.items()])
        print("Equivalent SQL:")
        print(f"  MERGE INTO {target_table} AS t\n  USING (source batch, deduplicated on {keys}) AS s\n  ON {merge_condition}\n  WHEN MATCHED THEN UPDATE SET\n    {set_clause}\n  WHEN NOT MATCHED THEN INSERT *")

        # Open the target Delta table and give it the alias t
        (DeltaTable.forName(spark, target_table).alias("t")
            # Merge the source data (alias s) into the target using the key condition
            .merge(source_df.alias("s"), merge_condition)
            # When a key already exists in the target, update its columns
            .whenMatchedUpdate(set=update_set)
            # When a key does not exist in the target, insert the full row
            .whenNotMatchedInsertAll()
            # Run the MERGE
            .execute())

# Print the result of the write from the Delta table history (rows inserted/updated/written)
print("-" * 80)
last_op = DeltaTable.forName(spark, target_table).history(1).select("version", "operation", "operationMetrics").first()
print(f"Delta version : {last_op['version']}")
print(f"Operation     : {last_op['operation']}")
print("Metrics       :")
for metric_name, metric_value in sorted(last_op["operationMetrics"].items()):
    print(f"  {metric_name:<35} {metric_value}")
print(f"Rows in target now : {spark.table(target_table).count()}")

# Print a message confirming which load type finished and for which table
print(f"Finished at   : {datetime.now()}")
print(f"{load_type_normalized} load completed for {target_table}")
print("=" * 80)

In [0]:
# Count the number of rows processed in this batch
record_count = updated_df.count()
# End the notebook and return the record count to the calling job or notebook
dbutils.notebook.exit({"record_count": record_count})